## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [1]:
import seaborn as sns

titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
718,0,3,male,NaN,0,0,15.5000,Q,Third,man,True,NaN,Queenstown,no,True
226,1,2,male,19.0,0,0,10.5000,S,Second,man,True,NaN,Southampton,yes,True
137,0,1,male,37.0,1,0,53.1000,S,First,man,True,C,Southampton,no,False
608,1,2,female,22.0,1,2,41.5792,C,Second,woman,False,NaN,Cherbourg,yes,False
272,1,2,female,41.0,0,1,19.5000,S,Second,woman,False,NaN,Southampton,yes,False


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [ ]:
nans = titanic_data.isnull().sum()
print(nans)

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [8]:
titanic_data = titanic_data.dropna(thresh=0.5 * len(titanic_data), axis=1)
titanic_data = titanic_data.dropna(thresh=0.5 * len(titanic_data.columns), axis=0)
print(titanic_data["age"].isnull().sum())

0


### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [9]:
for who in ["man", "woman", "child"]:
    median = round(titanic_data[titanic_data["who"] == who]["age"].median())
    titanic_data.loc[(titanic_data["who"] == who) & (titanic_data["age"].isnull()), "age"] = median

### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [ ]:
titanic_data = titanic_data.dropna(thresh=len(titanic_data.columns) - 1, axis=0)

Есть ли пропуски в таблице? False


### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [ ]:
city = titanic_data["embark_town"].value_counts().idxmax()

Город, из которого больше всего пассажиров: Southampton


### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [ ]:
k_surv = titanic_data["survived"].mean() * 100

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [ ]:
survivors_by_city = titanic_data[titanic_data["survived"] == 1]["embark_town"].value_counts()

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [ ]:
survived = titanic_data[titanic_data["survived"] == 1]["pclass"].value_counts()
total = titanic_data["pclass"].value_counts()
result_series = round(survived / total * 100, 2)

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [ ]:
rich = titanic_data[titanic_data["fare"] >= 100]
rich_survived = round(rich["survived"].mean() * 100, 2)

### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [ ]:
lonely = len(titanic_data[titanic_data["alone"] == 1 & titanic_data["who"] == "child"])

Какие выводы вы можете сделать о выживших пассажирах Титаника? 